# Bài 1 — Luồng dữ liệu với Kafka
`bai1_kafka.ipynb`

---

Xây dựng lớp tiếp nhận dữ liệu (ingestion layer) cho một sàn thương mại điện tử. Mỗi giao dịch được ghi nhận dưới dạng sự kiện và đưa vào Kafka để các hệ thống phía sau xử lý.

**Mục tiêu:**
- Hiểu mô hình publish/subscribe trong thực tế
- Viết producer gửi sự kiện lên Kafka topic
- Viết consumer đọc và xử lý sự kiện từ topic
- Quan sát cơ chế partition và offset qua code

## Phần A — Kiểm tra môi trường

In [ ]:
import subprocess
result = subprocess.run(
    ["docker", "exec", "kafka", "kafka-topics.sh", "--list", "--bootstrap-server", "localhost:9092"],
    capture_output=True, text=True
)
if result.returncode == 0:
    print("✓ Kafka đang chạy")
    print(f"  Topics: {[t for t in result.stdout.strip().split(chr(10)) if t]}")
else:
    print("✗ Không kết nối được Kafka — kiểm tra: docker compose ps")

In [ ]:
import os, uuid, json, random, time
from datetime import datetime
from kafka import KafkaProducer, KafkaConsumer

KAFKA_BOOTSTRAP = os.environ.get("KAFKA_BOOTSTRAP_SERVERS", "kafka:9093")
TOPIC_NAME      = "transactions"

# Kiểm tra topic
from kafka import KafkaConsumer as _KC
_c = _KC(bootstrap_servers=KAFKA_BOOTSTRAP)
partitions = _c.partitions_for_topic(TOPIC_NAME)
_c.close()
print(f"Topic '{TOPIC_NAME}': {len(partitions) if partitions else '✗ chưa tồn tại'} partitions")

## Phần B — Producer

| Trường | Kiểu | Mô tả |
|---|---|---|
| `transaction_id` | str | UUID duy nhất |
| `user_id` | str | `user_001` đến `user_500` |
| `product_id` | str | `prod_001` đến `prod_200` |
| `amount` | float | Làm tròn 2 chữ số |
| `category` | str | electronics / fashion / food / books / home / sports |
| `timestamp` | str | ISO 8601 |
| `is_fraud` | bool | True với xác suất 2% |

In [ ]:
CATEGORIES = ["electronics", "fashion", "food", "books", "home", "sports"]
AMOUNT_RANGE = {
    "electronics": (50.0, 5000.0), "fashion": (20.0, 800.0),
    "food": (5.0, 200.0),          "books":   (5.0, 150.0),
    "home": (15.0, 2000.0),        "sports":  (10.0, 1000.0),
}

In [ ]:
def generate_transaction() -> dict:
    # TODO: tạo và trả về dict theo schema trên
    pass


# Kiểm tra
sample = generate_transaction()
print(json.dumps(sample, indent=2, ensure_ascii=False))
assert isinstance(sample["transaction_id"], str) and len(sample["transaction_id"]) == 36
assert sample["user_id"].startswith("user_")
assert sample["product_id"].startswith("prod_")
assert isinstance(sample["amount"], float)
assert isinstance(sample["is_fraud"], bool)
assert "T" in sample["timestamp"]
print("✓ generate_transaction() hợp lệ")

In [ ]:
def run_producer(n: int = 200, delay: float = 0.05) -> dict:
    """Gửi n sự kiện lên topic. Trả về {"sent", "fraud", "errors"}."""
    # TODO: tạo KafkaProducer với value_serializer và key_serializer
    # TODO: gửi từng sự kiện, dùng user_id làm key
    # TODO: in tiến độ mỗi 50 message
    # TODO: flush() và close() trước khi kết thúc
    pass


print("Bắt đầu gửi dữ liệu...
")
result = run_producer(n=200, delay=0.05)

## Phần C — Consumer

In [ ]:
def run_consumer(max_messages: int = 50, timeout_ms: int = 8000) -> list:
    """Đọc tối đa max_messages message. Trả về list các dict đã đọc."""
    # TODO: tạo KafkaConsumer với group_id="analysis-group", auto_offset_reset="earliest"
    # TODO: in mỗi message theo định dạng:
    #       [HH:MM:SS]  {transaction_id}  |  {user_id}  |  {amount:,.0f}đ  |  {category}
    # TODO: đóng consumer và in tổng kết
    pass


print("Đọc dữ liệu từ Kafka...
")
received = run_consumer(max_messages=50)

## Phần D — Câu hỏi quan sát

Mở Kafka UI tại **http://localhost:8081**

**Câu 1:** Mở Kafka UI → Topics → `transactions` → Messages, quan sát cột Partition.
Các message phân bổ vào 3 partitions như thế nào? Tại sao?

*(Trả lời tại đây)*

In [ ]:
# Gợi ý: minh họa nguyên lý phân partition theo key
def simulate_partition(key: str, num_partitions: int = 3) -> int:
    return abs(hash(key)) % num_partitions

from collections import Counter
dist = Counter(simulate_partition(f"user_{i:03d}") for i in range(1, 501))
for p, cnt in sorted(dist.items()):
    print(f"Partition {p}: {cnt} users ({cnt/5:.1f}%)")

**Câu 2:** Nếu chạy 2 consumer cùng `group_id`, điều gì xảy ra? Thử và ghi lại kết quả.

*(Trả lời tại đây)*

**Câu 3:** Đổi `auto_offset_reset` sang `"latest"` — consumer có đọc được message cũ không? Tại sao?

*(Trả lời tại đây)*

## Kiểm tra đầu ra

In [ ]:
print("=" * 45)
print("KIỂM TRA ĐẦU RA BÀI 1")
print("=" * 45)

checks = []
txn = generate_transaction()
checks.append(("Schema đầy đủ 7 trường",
    all(k in txn for k in ["transaction_id","user_id","product_id","amount","category","timestamp","is_fraud"])))
checks.append(("amount là float, is_fraud là bool",
    isinstance(txn["amount"], float) and isinstance(txn["is_fraud"], bool)))
checks.append(("user_id và product_id đúng định dạng",
    txn["user_id"].startswith("user_") and txn["product_id"].startswith("prod_")))
checks.append(("Producer gửi ≥ 200 message", result.get("sent", 0) >= 200))
checks.append(("Không có lỗi khi gửi", result.get("errors", 0) == 0))
checks.append(("Consumer đọc được message", len(received) > 0))
fraud_rate = result["fraud"] / result["sent"] if result["sent"] > 0 else 0
checks.append((f"Fraud rate hợp lý ({fraud_rate:.1%})", 0.0 <= fraud_rate <= 0.10))

all_passed = True
for name, passed in checks:
    print(f"  [{'✓' if passed else '✗'}] {name}")
    if not passed: all_passed = False
print()
print("✓ Bài 1 hoàn thành!" if all_passed else "✗ Xem lại phần chưa qua.")